In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN=userdata.get('HF')

if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token is not set. Please save the token first.")

In [ ]:
import os
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer
from typing import Dict, List, cast
from tqdm import tqdm
from pathlib import Path
from datasets import load_dataset, DatasetDict

from subnet_model import SubnetLLM

In [ ]:
MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
CACHE_DIR = './cache'
EMBEDDING_LAYERS = 2
COHERENCE_LAYERS = 2
ADAPTATION_LAYERS = 2
COMPENSATION_LAYERS = 2
CONCATENATION_LAYERS = 1

DATASET_PATH = 'SergiuNistor/yoro-train'

BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 3e-4
NUM_EPOCHS = 10
MAX_GRAD_NORM = 1.0
WEIGHT_DECAY = 0.01

SANITY_CHECK_EPOCHS = 1000
SANITY_CHECK_OUTPUT_TOKENS = 10

DEVICE = 'cuda'
DTYPE = torch.bfloat16
LOSS_PLOT_PATH = 'loss.png'

# Checkpoint settings
SAVE_STEPS = 10000 / GRADIENT_ACCUMULATION_STEPS  # Save checkpoint every N steps
SAVE_DIR = Path('/content/drive/MyDrive/YORO/checkpoints')
CHECKPOINT_SAVE_NAME = 'checkpoint'  # Base name for saved checkpoints (will append step number)
CHECKPOINT_PATH = '/content/drive/MyDrive/YORO/checkpoints/checkpoint_latest.pt'  # Set to checkpoint path to resume training (e.g., './checkpoints/training_checkpoint_step_5000.pt')

In [ ]:
def collate_batch(batch, pad_token_id: int, device: str):
    """Collate a batch of samples with left padding."""
    # Extract data from batch
    input_ids_list = [torch.tensor(sample['input_token_ids'], dtype=torch.long) for sample in batch]
    logprob_token_ids_list = [torch.tensor(sample['logprob_token_ids'], dtype=torch.long) for sample in batch]
    logprob_values_list = [torch.tensor(sample['logprob_values'], dtype=torch.float32) for sample in batch]

    # Get prompt lengths (input) and sequence lengths (input + output)
    prompt_lengths = [len(ids) for ids in input_ids_list]
    num_outputs = [logprobs.shape[0] for logprobs in logprob_token_ids_list]
    total_lengths = [p + o for p, o in zip(prompt_lengths, num_outputs)]

    max_total_length = max(total_lengths)
    max_output_length = max(num_outputs)
    batch_size = len(batch)

    # Initialize tensors with padding
    padded_input_ids = torch.full((batch_size, max_total_length), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_total_length), dtype=torch.long)

    # Teacher distributions: store token IDs and probabilities
    teacher_token_ids = torch.full((batch_size, max_output_length, logprob_token_ids_list[0].shape[1]),
                                    pad_token_id, dtype=torch.long)
    teacher_probs = torch.zeros((batch_size, max_output_length, logprob_values_list[0].shape[1]), dtype=torch.float32)
    output_mask = torch.zeros((batch_size, max_output_length), dtype=torch.bool)

    actual_prompt_lengths = []

    for i in range(batch_size):
        prompt_len = prompt_lengths[i]
        output_len = num_outputs[i]
        total_len = total_lengths[i]

        # Left padding: pad_length tokens of padding, then prompt, then ground truth outputs
        pad_length = max_total_length - total_len

        # Fill in input sequence (prompt only)
        padded_input_ids[i, pad_length:pad_length + prompt_len] = input_ids_list[i]

        # Fill in ground truth tokens for teacher forcing (top-1 from logprobs)
        ground_truth_tokens = logprob_token_ids_list[i][:, 0]  # [output_len]
        padded_input_ids[i, pad_length + prompt_len:pad_length + total_len] = ground_truth_tokens

        # Attention mask (1 for real tokens, 0 for padding)
        attention_mask[i, pad_length:] = 1

        # Store prompt length (accounting for padding)
        actual_prompt_lengths.append(pad_length + prompt_len)

        # Fill teacher distributions
        teacher_token_ids[i, :output_len] = logprob_token_ids_list[i]
        teacher_probs[i, :output_len] = logprob_values_list[i]
        output_mask[i, :output_len] = True

    return {
        'input_ids': padded_input_ids.to(device),
        'attention_mask': attention_mask.to(device),
        'prompt_lengths': torch.tensor(actual_prompt_lengths, dtype=torch.long, device=device),
        'teacher_token_ids': teacher_token_ids.to(device),
        'teacher_probs': teacher_probs.to(device),
        'output_mask': output_mask.to(device)
    }

def compute_distillation_loss(logits, teacher_token_ids, teacher_probs, output_mask, vocab_size):
    """
    Compute KL divergence loss between student and teacher distributions.

    Args:
        logits: [batch_size, num_outputs, vocab_size] student logits (already extracted for output positions)
        teacher_token_ids: [batch_size, num_outputs, top_k] teacher token IDs
        teacher_probs: [batch_size, num_outputs, top_k] teacher probabilities
        output_mask: [batch_size, num_outputs] mask for valid outputs (1 for real, 0 for padding)
        vocab_size: vocabulary size

    Returns:
        Average KL divergence loss (only over valid, non-padded positions)
    """
    batch_size, num_outputs, top_k = teacher_token_ids.shape

    # Convert student logits to log probabilities
    student_log_probs = F.log_softmax(logits, dim=-1)  # [batch_size, num_outputs, vocab_size]

    # Build teacher probability distribution
    teacher_dist = torch.zeros(batch_size, num_outputs, vocab_size, device=logits.device, dtype=torch.float32)

    for i in range(batch_size):
        for j in range(num_outputs):
            if output_mask[i, j]:
                token_ids = teacher_token_ids[i, j]
                probs = teacher_probs[i, j]
                teacher_dist[i, j, token_ids] = probs

    # Teacher log probs
    teacher_log_probs = torch.log(teacher_dist + 1e-10)

    # KL divergence: sum(teacher * (log(teacher) - log(student)))
    kl_div = torch.sum(teacher_dist * (teacher_log_probs - student_log_probs), dim=-1)  # [batch_size, num_outputs]

    # Apply mask and compute mean (only over valid positions)
    masked_kl = kl_div * output_mask.float()
    loss = masked_kl.sum() / output_mask.sum().clamp(min=1)  # Avoid division by zero

    return loss


def save_training_checkpoint(
    model: SubnetLLM,
    optimizer: torch.optim.Optimizer,
    scaler: torch.GradScaler,
    epoch: int,
    global_step: int,
    train_losses: List[float],
    val_losses: List[float],
    best_val_loss: float,
    model_config: Dict,
    dataset_position: int = 0
) -> None:
    """Save complete training state for resuming."""
    checkpoint = {
        'epoch': epoch,
        'global_step': global_step,
        'dataset_position': dataset_position,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        'rng_state': torch.get_rng_state().cpu(),  # Ensure on CPU
        'cuda_rng_state': torch.cuda.get_rng_state().cpu(),  # Ensure on CPU
        'config': {
            'model_name': model_config['model_name'],
            'embedding_layers': model_config['embedding_layers'],
            'coherence_layers': model_config['coherence_layers'],
            'adaptation_layers': model_config['adaptation_layers'],
            'compensation_layers': model_config['compensation_layers'],
            'concatenation_layers': model_config['concatenation_layers'],
        }
    }
    save_path = SAVE_DIR / f'{CHECKPOINT_SAVE_NAME}_step_{global_step}.pt'
    torch.save(checkpoint, save_path)
    print(f"\nCheckpoint saved to {save_path}")

    # Also save as latest for easy resumption
    latest_path = SAVE_DIR / f'{CHECKPOINT_SAVE_NAME}_latest.pt'
    torch.save(checkpoint, latest_path)


def load_training_checkpoint(
    checkpoint_path: str,
    model: SubnetLLM,
    optimizer: torch.optim.Optimizer,
    scaler: torch.GradScaler
) -> Dict:
    """Load complete training state to resume training."""
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

    # Handle compiled model state dicts (strip _orig_mod. prefix if present)
    model_state_dict = checkpoint['model_state_dict']
    if any(key.startswith('_orig_mod.') for key in model_state_dict.keys()):
        print("Detected compiled model checkpoint, stripping _orig_mod. prefix...")
        model_state_dict = {
            key.replace('_orig_mod.', ''): value
            for key, value in model_state_dict.items()
        }

    model.load_state_dict(model_state_dict)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])

    # Restore RNG states for reproducibility (ensure they're on CPU)
    rng_state = checkpoint['rng_state']
    cuda_rng_state = checkpoint['cuda_rng_state']

    # Move to CPU if needed
    if rng_state.device.type != 'cpu':
        rng_state = rng_state.cpu()
    if cuda_rng_state.device.type != 'cpu':
        cuda_rng_state = cuda_rng_state.cpu()

    torch.set_rng_state(rng_state)
    torch.cuda.set_rng_state(cuda_rng_state)

    print(f"Resumed from epoch {checkpoint['epoch'] + 1}, step {checkpoint['global_step']}")

    return {
        'start_epoch': checkpoint['epoch'],
        'global_step': checkpoint['global_step'],
        'dataset_position': checkpoint.get('dataset_position', 0),
        'train_losses': checkpoint['train_losses'],
        'val_losses': checkpoint['val_losses'],
        'best_val_loss': checkpoint['best_val_loss']
    }


def train_epoch(
    model: SubnetLLM,
    dataset: DatasetDict,
    optimizer: torch.optim.Optimizer,
    scaler: torch.GradScaler,
    pad_token_id: int,
    vocab_size: int,
    tokenizer: AutoTokenizer,
    epoch: int,
    global_step: int,
    train_losses: List[float],
    val_losses: List[float],
    best_val_loss: float,
    model_config: Dict,
    start_step: int = 0
) -> tuple[float, int]:
    """Train for one epoch, saving checkpoints periodically. Returns (avg_loss, final_global_step)."""
    model.train()
    total_loss = 0.0
    num_batches = 0

    # Create generator with deterministic seed based on epoch for reproducible shuffling
    # This ensures each epoch has the same data order regardless of when we resume
    generator = torch.Generator()
    generator.manual_seed(42 + epoch)  # Different seed for each epoch, but deterministic

    # Create dataloader with deterministic shuffling
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda batch: collate_batch(batch, pad_token_id, DEVICE),
        num_workers=0,
        generator=generator  # Use deterministic generator for shuffling
    )

    # Initialize progress bar
    progress_bar = tqdm(dataloader, desc="Training")
    optimizer.zero_grad()

    for step, batch in enumerate(progress_bar):
        # Skip steps if resuming mid-epoch, but update progress bar
        if step < start_step:
            progress_bar.update()
            continue

        # Forward pass with teacher forcing
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            # Get per-sequence prompt lengths
            prompt_lengths = batch['prompt_lengths']  # [batch_size]
            batch_size = prompt_lengths.shape[0]
            seq_length = batch['input_ids'].shape[1]

            # We need to handle variable prompt lengths properly
            # For now, use minimum prompt length for teacher forcing
            # (This means some sequences will have their early outputs treated as part of prompt)
            min_prompt_length = prompt_lengths.min().item()

            logits, _ = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                use_teacher_forcing=True,
                prompt_length=min_prompt_length
            )

            # Extract logits for output positions
            # Logits at position i predict token at position i+1
            # For each sequence, we want to predict tokens at positions [prompt_length:seq_length]
            # So we need logits from positions [prompt_length-1:seq_length-1]

            # Build output logits and masks accounting for variable prompt lengths
            max_output_length = batch['teacher_token_ids'].shape[1]
            output_logits_list = []
            output_mask_adjusted = batch['output_mask'].clone()

            for i in range(batch_size):
                seq_prompt_len = prompt_lengths[i].item()

                # How many positions were incorrectly treated as prompt due to using min_prompt_length?
                extra_prompt_positions = seq_prompt_len - min_prompt_length

                # Extract logits for this sequence's outputs
                # Start from position seq_prompt_len-1 (which predicts token at seq_prompt_len)
                start_idx = seq_prompt_len - 1
                end_idx = start_idx + max_output_length

                # Clamp to valid range
                end_idx = min(end_idx, seq_length)
                seq_output_logits = logits[i, start_idx:end_idx, :]  # [actual_output_len, vocab_size]

                # Pad if needed to match max_output_length
                if seq_output_logits.shape[0] < max_output_length:
                    padding = torch.zeros(
                        (max_output_length - seq_output_logits.shape[0], vocab_size),
                        device=logits.device,
                        dtype=logits.dtype
                    )
                    seq_output_logits = torch.cat([seq_output_logits, padding], dim=0)

                output_logits_list.append(seq_output_logits)

                # Mask out positions that were incorrectly included due to min_prompt_length
                if extra_prompt_positions > 0:
                    output_mask_adjusted[i, :extra_prompt_positions] = False

            output_logits = torch.stack(output_logits_list, dim=0)  # [batch_size, max_output_length, vocab_size]

            loss = compute_distillation_loss(
                output_logits,
                batch['teacher_token_ids'],
                batch['teacher_probs'],
                output_mask_adjusted,
                vocab_size
            )

        # Backward pass
        scaler.scale(loss).backward()

        # Gradient accumulation
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            # Increment global step after optimizer step
            global_step += 1

            # Save checkpoint every SAVE_STEPS
            if global_step % SAVE_STEPS == 0:
                save_training_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    scaler=scaler,
                    epoch=epoch,
                    global_step=global_step,
                    train_losses=train_losses,
                    val_losses=val_losses,
                    best_val_loss=best_val_loss,
                    model_config=model_config,
                    dataset_position=step
                )

        total_loss += loss.item()
        num_batches += 1

        progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'step': global_step})

        # Sanity check
        if step % SANITY_CHECK_EPOCHS == 0 and step > 0:
            model.eval()
            with torch.no_grad():
                test_conversations = [{
                    'role': 'system',
                    'content': 'You are a helpful assistant'
                }, {
                    'role': 'user',
                    'content': 'How are you?'
                }]
                test_chat_conversations = tokenizer.apply_chat_template(
                    test_conversations,
                    add_generation_prompt=True,
                    tokenize=False
                )
                prompt_ids = tokenizer(
                    test_chat_conversations,
                    return_tensors="pt"
                )['input_ids'].to(DEVICE)

                attention_mask = torch.ones_like(prompt_ids)
                cached_reasoning_outputs = [None]
                generated = prompt_ids

                for _ in range(SANITY_CHECK_OUTPUT_TOKENS):
                    logits, cached_reasoning_outputs = model(
                        input_ids=generated,
                        cached_reasoning_outputs=cached_reasoning_outputs,
                        attention_mask=attention_mask,
                        use_teacher_forcing=False
                    )

                    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                    generated = torch.cat([generated, next_token], dim=1)
                    attention_mask = torch.ones_like(generated)

                    if next_token.item() == pad_token_id:
                        break

                print("\nModel response:", tokenizer.decode(generated[0].tolist(), skip_special_tokens=True))
            model.train()

    # Final gradient update if there are remaining gradients
    if num_batches % GRADIENT_ACCUMULATION_STEPS != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        global_step += 1

    return (total_loss / num_batches if num_batches > 0 else 0.0, global_step)


def validate(
    model: SubnetLLM,
    dataset: DatasetDict,
    pad_token_id: int,
    vocab_size: int
) -> float:
    model.eval()
    total_loss = 0.0
    num_batches = 0

    # Create dataloader
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda batch: collate_batch(batch, pad_token_id, DEVICE),
        num_workers=0
    )

    progress_bar = tqdm(dataloader, desc="Validating")

    with torch.no_grad():
        for batch in progress_bar:
            with torch.autocast(device_type='cuda', dtype=DTYPE):
                prompt_lengths = batch['prompt_lengths']
                batch_size = prompt_lengths.shape[0]
                seq_length = batch['input_ids'].shape[1]
                min_prompt_length = prompt_lengths.min().item()

                logits, _ = model(
                    input_ids=batch['input_ids'],
                    attention_mask=batch['attention_mask'],
                    use_teacher_forcing=True,
                    prompt_length=min_prompt_length
                )

                # Extract logits for output positions with proper masking
                max_output_length = batch['teacher_token_ids'].shape[1]
                output_logits_list = []
                output_mask_adjusted = batch['output_mask'].clone()

                for i in range(batch_size):
                    seq_prompt_len = prompt_lengths[i].item()
                    extra_prompt_positions = seq_prompt_len - min_prompt_length

                    start_idx = seq_prompt_len - 1
                    end_idx = min(start_idx + max_output_length, seq_length)
                    seq_output_logits = logits[i, start_idx:end_idx, :]

                    if seq_output_logits.shape[0] < max_output_length:
                        padding = torch.zeros(
                            (max_output_length - seq_output_logits.shape[0], vocab_size),
                            device=logits.device,
                            dtype=logits.dtype
                        )
                        seq_output_logits = torch.cat([seq_output_logits, padding], dim=0)

                    output_logits_list.append(seq_output_logits)

                    if extra_prompt_positions > 0:
                        output_mask_adjusted[i, :extra_prompt_positions] = False

                output_logits = torch.stack(output_logits_list, dim=0)

                loss = compute_distillation_loss(
                    output_logits,
                    batch['teacher_token_ids'],
                    batch['teacher_probs'],
                    output_mask_adjusted,
                    vocab_size
                )

            total_loss += loss.item()
            num_batches += 1

            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / num_batches if num_batches > 0 else 0.0


def save_checkpoint(
    model: SubnetLLM,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    loss: float,
    model_config: Dict
) -> None:
    """Save best model checkpoint (kept for backward compatibility)."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'config': {
            'model_name': model_config['model_name'],
            'embedding_layers': model_config['embedding_layers'],
            'coherence_layers': model_config['coherence_layers'],
            'adaptation_layers': model_config['adaptation_layers'],
            'compensation_layers': model_config['compensation_layers'],
            'concatenation_layers': model_config['concatenation_layers'],
        }
    }
    save_path = SAVE_DIR / 'best_model.pt'
    torch.save(checkpoint, save_path)
    print(f"Best model checkpoint saved to {save_path}")


def plot_and_save_losses(
    train_losses: List[float],
    val_losses: List[float]
) -> None:
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_losses) + 1)

    plt.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)

    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plt.savefig(LOSS_PLOT_PATH, dpi=300, bbox_inches='tight')
    print(f"\nLoss plot saved to {LOSS_PLOT_PATH}")

    plt.show()

In [ ]:
assert torch.cuda.is_available()

torch.cuda.empty_cache()

SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("Loading datasets...")
dataset = load_dataset(DATASET_PATH, num_proc=os.cpu_count())
train_dataset = cast(DatasetDict, dataset['train'])
val_dataset = cast(DatasetDict, dataset['validation'])

print(f"Train sequences: {len(train_dataset)}")
print(f"Val sequences: {len(val_dataset)}")
print(f"Total sequences: {len(train_dataset) + len(val_dataset)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
pad_token_id = tokenizer.pad_token_id
vocab_size = tokenizer.vocab_size

print("\nInitializing model...")
model = SubnetLLM(
    model_name=MODEL_NAME,
    cache_dir=CACHE_DIR,
    embedding_layers=EMBEDDING_LAYERS,
    coherence_layers=COHERENCE_LAYERS,
    adaptation_layers=ADAPTATION_LAYERS,
    compensation_layers=COMPENSATION_LAYERS,
    concatenation_layers=CONCATENATION_LAYERS,
    device=DEVICE,
    dtype=DTYPE
)

# Count all parameters vs trainable
all_params = sum(p.numel() for p in model.parameters())
trainable_params = [p for p in model.parameters() if p.requires_grad]
trainable_count = sum(p.numel() for p in trainable_params)

print(f"Total parameters: {all_params:,}")
print(f"Trainable parameters: {trainable_count:,}")
print(f"Frozen parameters: {all_params - trainable_count:,}")

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Initialize GradScaler for mixed precision training
scaler = torch.GradScaler('cuda', enabled=(DTYPE not in [torch.float16, torch.bfloat16]))

model_config = {
    'model_name': MODEL_NAME,
    'embedding_layers': EMBEDDING_LAYERS,
    'coherence_layers': COHERENCE_LAYERS,
    'adaptation_layers': ADAPTATION_LAYERS,
    'compensation_layers': COMPENSATION_LAYERS,
    'concatenation_layers': CONCATENATION_LAYERS,
}

# Initialize training state
start_epoch = 0
global_step = 0
best_val_loss = float('inf')
train_losses = []
val_losses = []
resume_from_batch = 0  # Batch to resume from within first epoch

# Load checkpoint if specified (BEFORE compiling)
if CHECKPOINT_PATH is not None:
    checkpoint_state = load_training_checkpoint(
        checkpoint_path=CHECKPOINT_PATH,
        model=model,
        optimizer=optimizer,
        scaler=scaler
    )
    start_epoch = checkpoint_state['start_epoch']
    global_step = checkpoint_state['global_step']
    best_val_loss = checkpoint_state['best_val_loss']
    train_losses = checkpoint_state['train_losses']
    val_losses = checkpoint_state['val_losses']
    # Skip batches that were already processed (dataset_position is the last batch we processed)
    resume_from_batch = checkpoint_state['dataset_position'] + 1
    print(f"Continuing training from epoch {start_epoch + 1}, global step {global_step}")
    print(f"Will skip first {resume_from_batch} batches in epoch {start_epoch + 1}")
    print(f"Best validation loss so far: {best_val_loss:.4f}")

# Compile model AFTER loading checkpoint
print("\nCompiling model...")
model = cast(SubnetLLM, torch.compile(model))

print("\nStarting training...")

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")

    # Use resume_from_batch only on the first epoch after resuming, then reset to 0
    current_start_step = resume_from_batch if epoch == start_epoch else 0

    train_loss, global_step = train_epoch(
        model=model,
        dataset=train_dataset,
        optimizer=optimizer,
        scaler=scaler,
        pad_token_id=pad_token_id,
        vocab_size=vocab_size,
        tokenizer=tokenizer,
        epoch=epoch,
        global_step=global_step,
        train_losses=train_losses,
        val_losses=val_losses,
        best_val_loss=best_val_loss,
        model_config=model_config,
        start_step=current_start_step
    )

    # Reset resume_from_batch after first epoch
    resume_from_batch = 0

    print(f"Training loss: {train_loss:.4f}")
    train_losses.append(train_loss)

    val_loss = validate(
        model=model,
        dataset=val_dataset,
        pad_token_id=pad_token_id,
        vocab_size=vocab_size
    )
    print(f"Validation loss: {val_loss:.4f}")
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            loss=val_loss,
            model_config=model_config
        )

print("\nTraining completed!")

plot_and_save_losses(
    train_losses=train_losses,
    val_losses=val_losses
)